<a href="https://colab.research.google.com/github/ilyaaannn/justchill/blob/main/MODEL_ML_AQUAPONIC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MACHINE LEARNING K-NN

**Inisialisasi Sistem dan Manajemen Library**

Bagian pertama ini adalah pondasi dari seluruh sistem machine learning yang dibangun. Tujuan utamanya adalah memastikan semua *dependency* tersedia di runtime Google Colab.

*   **Instalasi Library Eksternal:** Menggunakan `!pip install` untuk memuat library yang tidak tersedia secara *default*, seperti `scikit-fuzzy` untuk pemrosesan logika fuzzy dan `imblearn` untuk menangani ketidakseimbangan data (*imbalanced data*).
*   **Library Data Science:** Memuat library standar seperti `Pandas` (manipulasi tabel), `NumPy` (komputasi numerik), serta `Matplotlib` dan `Seaborn` untuk visualisasi data.
*   **Modul Machine Learning:** Memuat modul dari `scikit-learn` untuk pemisahan data, standarisasi fitur, algoritma K-Nearest Neighbors (KNN), dan metrik evaluasi.
*   **Environment Setup:** Mengaktifkan filter peringatan agar output tetap bersih dan melakukan *mounting* Google Drive untuk akses penyimpanan dataset serta model biner (.pkl).

In [ ]:
!pip install kagglehub scikit-learn pandas numpy joblib imblearn matplotlib seaborn scikit-fuzzy

import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import skfuzzy as fuzz
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
import joblib
import pickle
import os
import shutil
import warnings as warning
warning.filterwarnings("ignore")

**Integrasi Data (Data Fusion) dari Sumber Berbeda**

Langkah ini merupakan implementasi dari konsep *Data Fusion*, di mana sistem menggabungkan informasi dari beberapa sumber sensor untuk membentuk dataset yang komprehensif.

*   **Pemuatan Dataset:** Data diambil langsung dari Kaggle menggunakan API `kagglehub`. Terdapat dua dataset utama: dataset `samahfetouh/cleaned-aquaponics-pond-dataset` (untuk parameter *Dissolved Oxygen*) dan dataset `bobsis/small-aquaculture-fishpond` (untuk pH, TDS, dan Suhu).
*   **Pembersihan Data:** Dilakukan proses `dropna()` untuk memastikan tidak ada nilai kosong yang masuk ke tahap pemodelan.
*   **Teknik Random Sampling:** Menggunakan `sample()` untuk mengambil 5.000 baris data secara acak dari kedua sumber. Teknik ini memastikan distribusi data tetap representatif meskipun berasal dari sumber yang berbeda.
*   **Konkatenasi:** Data digabungkan secara horizontal untuk menghasilkan satu DataFrame (`df`) yang memiliki 4 parameter kualitas air yang saling berkorelasi.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

SEED = 42
MAX_ROWS = 5000

# ====== LOAD DATASET 1: SENSOR-BASED AQUAPONICS ======
print("\nLOADING DATASET 1: Sensor-Based Aquaponics Fish Pond")
try:
    df1 = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "samahfetouh/cleaned-aquaponics-pond-dataset",
        "IoTPond3.csv"
    )
    print(f"Dataset 1: {df1.shape[0]} rows, {df1.shape[1]} columns")
    print(f"   Columns: {list(df1.columns)}")
except Exception as e:
    print(f"❌ Error: {e}")
    raise

# ====== LOAD DATASET 2: SMALL AQUACULTURE FISHPOND ======
print("\nLOADING DATASET 2: Small Aquaculture Fishpond")
try:
    df2 = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "bobsis/small-aquaculture-fishpond",
        "pond_iot_2023_raw.csv"
    )
    print(f"Dataset 2: {df2.shape[0]} rows, {df2.shape[1]} columns")
    print(f"   Columns: {list(df2.columns)}")
except Exception as e:
    print(f"❌ Error: {e}")
    raise

# 1. Ekstrak dan bersihkan nilai kosong (NaN) dari Dataset 1
df1_extracted = df1[['disolved_oxg']].copy().dropna()
df1_extracted.columns = ['DO(mg/l)']

# 2. Ekstrak dan bersihkan nilai kosong (NaN) dari Dataset 2
df2_extracted = df2[['water_pH', 'TDS', 'water_temp']].copy().dropna()
df2_extracted.columns = ['pH(ph_units)', 'TDS(ppm)', 'Temp(cel)']

# 3. Pengambilan Sampel Acak (Random Sampling)
# Menggunakan replace=True agar aman jika data asli < MAX_ROWS
df1_sample = df1_extracted.sample(n=MAX_ROWS, random_state=SEED, replace=True).reset_index(drop=True)
df2_sample = df2_extracted.sample(n=MAX_ROWS, random_state=SEED, replace=True).reset_index(drop=True)

# 4. Penggabungan Horizontal (Data Fusion)
df3_combined = pd.concat([df1_sample, df2_sample], axis=1)

if len(df3_combined) > MAX_ROWS:
    df3_combined = df3_combined.sample(n=MAX_ROWS, random_state=SEED).reset_index(drop=True)
print(f"✓ Dataset gabungan dibatasi ke {MAX_ROWS:,} baris.")

print(f"\n✓ Dataset 3 Berhasil Dibuat! (Data Fusion via Sampling)")
print(f"    ✓ Shape      : {df3_combined.shape[0]:,} baris × {df3_combined.shape[1]} kolom")
print(f"    ✓ Columns    : {list(df3_combined.columns)}")
print(f"    ✓ Missing values per kolom:")
for col in df3_combined.columns:
    print(f"       - {col}: {df3_combined[col].isnull().sum()}")

# Simpan sebagai dataframe utama
df = df3_combined.copy()
print("\n--- Info Dataset ---")
print(df.info())
print("\n--- Deskripsi Statistik ---")
print(df.describe())

**Pengembangan Sistem Logika Fuzzy (Knowledge-Based Labeling)**

Karena dataset awal bersifat *unlabeled* (tidak memiliki target), bagian ini berfungsi sebagai sistem pakar otomatis yang memberikan label berdasarkan standar budidaya ikan.

*   **Fungsi Keanggotaan (*Membership Functions*):** Menggunakan fungsi trapesium (`trapmf`) untuk mendefinisikan batas-batas linguistik: **Ideal**, **Normal**, dan **Bahaya**. Batas-batas ini ditentukan berdasarkan literatur biologi ikan (misal: pH ideal antara 6.5 - 7.5).
*   **Fuzzy Inference System (FIS):** Mengimplementasikan logika Mamdani untuk menghitung derajat keanggotaan setiap baris data terhadap ketiga label tersebut.
*   **Mekanisme 'Hard Rule Check':** Ini adalah fitur keamanan kritis. Sebelum masuk ke perhitungan fuzzy, sistem melakukan pengecekan ambang batas ekstrem (seperti DO < 3 mg/l). Jika terdeteksi, sistem langsung melabeli 'Bahaya' tanpa mempedulikan parameter lain demi keselamatan ekosistem.

In [ ]:
# ===== PREPROCESSING FUZZY MEMBERSHIP FUNCTIONS =====
class FuzzyWaterQuality:

    def __init__(self):
        # Definisi range untuk setiap parameter
        self.do_range   = np.arange(0, 20.1, 0.1)
        self.ph_range   = np.arange(0, 14.1, 0.1)
        self.tds_range  = np.arange(0, 2000, 1)
        self.temp_range = np.arange(0, 50.1, 0.1)

        # Setup fuzzy membership functions
        self._setup_do_fuzzy()
        self._setup_ph_fuzzy()
        self._setup_tds_fuzzy()
        self._setup_temp_fuzzy()

    def _setup_do_fuzzy(self):
        self.do_bahaya = fuzz.trapmf(self.do_range, [0, 0, 3.0, 4.0])
        self.do_normal = fuzz.trapmf(self.do_range, [3.0, 4.0, 5.5, 6.0])
        self.do_ideal  = fuzz.trapmf(self.do_range, [5.5, 6.0, 20.0, 20.0])

    def _setup_ph_fuzzy(self):
        self.ph_bahaya_low  = fuzz.trapmf(self.ph_range, [0, 0, 5.5, 5.7])
        self.ph_bahaya_high = fuzz.trapmf(self.ph_range, [8.3, 8.5, 14.0, 14.0])
        self.ph_normal_low  = fuzz.trapmf(self.ph_range, [5.5, 5.7, 6.3, 6.5])
        self.ph_normal_high = fuzz.trapmf(self.ph_range, [7.5, 7.7, 8.3, 8.5])
        self.ph_ideal       = fuzz.trapmf(self.ph_range, [6.3, 6.5, 7.5, 7.7])

    def _setup_tds_fuzzy(self):
        self.tds_ideal  = fuzz.trapmf(self.tds_range, [0, 0, 700, 800])
        self.tds_normal = fuzz.trapmf(self.tds_range, [700, 800, 1200, 1300])
        self.tds_bahaya = fuzz.trapmf(self.tds_range, [1200, 1300, 2000, 2000])

    def _setup_temp_fuzzy(self):
        self.temp_bahaya_low  = fuzz.trapmf(self.temp_range, [0, 0, 20.0, 21.0])
        self.temp_bahaya_high = fuzz.trapmf(self.temp_range, [33.0, 34.0, 50.0, 50.0])
        self.temp_normal_low  = fuzz.trapmf(self.temp_range, [20.0, 21.0, 24.0, 25.0])
        self.temp_normal_high = fuzz.trapmf(self.temp_range, [30.0, 31.0, 33.0, 34.0])
        self.temp_ideal       = fuzz.trapmf(self.temp_range, [24.0, 25.0, 30.0, 31.0])

    def get_membership_values(self, do, ph, tds, temp):
        do   = np.clip(do, self.do_range[0], self.do_range[-1])
        ph   = np.clip(ph, self.ph_range[0], self.ph_range[-1])
        tds  = np.clip(tds, self.tds_range[0], self.tds_range[-1])
        temp = np.clip(temp, self.temp_range[0], self.temp_range[-1])

        memberships = {
            'do_ideal': fuzz.interp_membership(self.do_range, self.do_ideal, do),
            'do_normal': fuzz.interp_membership(self.do_range, self.do_normal, do),
            'do_bahaya': fuzz.interp_membership(self.do_range, self.do_bahaya, do),
            'ph_ideal': fuzz.interp_membership(self.ph_range, self.ph_ideal, ph),
            'ph_normal_low': fuzz.interp_membership(self.ph_range, self.ph_normal_low, ph),
            'ph_normal_high': fuzz.interp_membership(self.ph_range, self.ph_normal_high, ph),
            'ph_bahaya_low': fuzz.interp_membership(self.ph_range, self.ph_bahaya_low, ph),
            'ph_bahaya_high': fuzz.interp_membership(self.ph_range, self.ph_bahaya_high, ph),
            'tds_ideal': fuzz.interp_membership(self.tds_range, self.tds_ideal, tds),
            'tds_normal': fuzz.interp_membership(self.tds_range, self.tds_normal, tds),
            'tds_bahaya': fuzz.interp_membership(self.tds_range, self.tds_bahaya, tds),
            'temp_ideal': fuzz.interp_membership(self.temp_range, self.temp_ideal, temp),
            'temp_normal_low': fuzz.interp_membership(self.temp_range, self.temp_normal_low, temp),
            'temp_normal_high': fuzz.interp_membership(self.temp_range, self.temp_normal_high, temp),
            'temp_bahaya_low': fuzz.interp_membership(self.temp_range, self.temp_bahaya_low, temp),
            'temp_bahaya_high': fuzz.interp_membership(self.temp_range, self.temp_bahaya_high, temp),
        }
        return memberships

    # Agregasi dengan bobot prioritas
    def apply_fuzzy_rules(self, memberships):
        bahaya_scores = [
            memberships['do_bahaya'],
            memberships['ph_bahaya_low'], memberships['ph_bahaya_high'],
            memberships['tds_bahaya'],
            memberships['temp_bahaya_low'], memberships['temp_bahaya_high']
        ]
        bahaya = max(bahaya_scores)
        if bahaya >= 0.5:
            bahaya = min(1.0, bahaya * 1.2)

        normal = max(
            memberships['do_normal'],
            memberships['ph_normal_low'], memberships['ph_normal_high'],
            memberships['tds_normal'],
            memberships['temp_normal_low'], memberships['temp_normal_high']
        )

        ideal_scores = [
            memberships['do_ideal'],
            memberships['ph_ideal'],
            memberships['tds_ideal'],
            memberships['temp_ideal']
        ]
        # Ideal harus memenuhi semua parameter (gunakan weighted average, bukan min)
        ideal = sum(ideal_scores) / len(ideal_scores)

        # Penalty: jika ada parameter di bawah threshold, turunkan ideal
        if any(score < 0.3 for score in ideal_scores):
            ideal = ideal * 0.4

        categories = {'bahaya': bahaya, 'normal': normal, 'ideal': ideal}
        return max(categories, key=categories.get), categories

    def hard_rule_check(self, do, ph, tds, temp):
        critical_conditions = [
            (do < 3.0, "DO kritis"), (do > 15.0, "DO supersaturasi"),
            (ph < 5.5, "pH asam"), (ph > 8.5, "pH basa"),
            (tds > 1300, "TDS tinggi"),
            (temp > 34.0, "Suhu tinggi"), (temp < 20.0, "Suhu rendah"),
        ]
        for is_critical, reason in critical_conditions:
            if is_critical:
                return 'bahaya', reason
        return None, None

# ====== FUZZY FEATURES ======
def create_fuzzy_labels_only(df):
    fuzzy_system = FuzzyWaterQuality()

    labels = []
    for idx, row in df.iterrows():
        do = row['DO(mg/l)']
        ph = row['pH(ph_units)']
        tds = row['TDS(ppm)']
        temp = row['Temp(cel)']

        hard_result, _ = fuzzy_system.hard_rule_check(do, ph, tds, temp)
        if hard_result == 'bahaya':
            label = 'bahaya'
        else:
            memberships = fuzzy_system.get_membership_values(do, ph, tds, temp)
            label, _ = fuzzy_system.apply_fuzzy_rules(memberships)
        labels.append(label)
    return np.array(labels), fuzzy_system

y_fuzzy, fuzzy_system = create_fuzzy_labels_only(df)
print("📊 Distribusi Label dari Fuzzy Rules:")
unique, counts = np.unique(y_fuzzy, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  {label}: {count} ({count/len(y_fuzzy)*100:.2f}%)")

# Tambahkan label ke dataframe
df['status'] = y_fuzzy
print(f"===== Total data: {len(y_fuzzy)} =====")

**Visualisasi Fungsi Keanggotaan Fuzzy**

Visualisasi ini bertujuan untuk memvalidasi secara visual apakah logika yang ditanamkan dalam kode sudah sesuai dengan rancangan sistem pakar.

*   **Plotting Parameter:** Setiap parameter (DO, pH, TDS, Suhu) divisualisasikan kurvanya.
*   **Interpretasi:** Area perpotongan antar kurva menunjukkan daerah transisi di mana sistem machine learning nantinya akan berperan lebih aktif dalam menentukan batas klasifikasi yang lebih presisi dibandingkan aturan manual.

In [ ]:
class FuzzyWaterQualityVisualization(FuzzyWaterQuality):
    def plot_all_membership_functions(self):
        fig, axes = plt.subplots(2, 2, figsize=(12, 7))
        fig.suptitle('Fungsi Keanggotaan Fuzzy untuk Parameter Kualitas Air', fontsize=16, fontweight='bold')

        # Plot DO
        axes[0, 0].plot(self.do_range, self.do_bahaya, 'r-', label='Bahaya')
        axes[0, 0].plot(self.do_range, self.do_normal, 'y-', label='Normal')
        axes[0, 0].plot(self.do_range, self.do_ideal, 'g-', label='Ideal')
        axes[0, 0].set_title('Dissolved Oxygen (DO)')
        axes[0, 0].legend()

        # Plot pH
        axes[0, 1].plot(self.ph_range, self.ph_bahaya_low, 'r-', label='Bahaya (Low)')
        axes[0, 1].plot(self.ph_range, self.ph_normal_low, 'y-', label='Normal (Low)')
        axes[0, 1].plot(self.ph_range, self.ph_ideal, 'g-', label='Ideal')
        axes[0, 1].plot(self.ph_range, self.ph_normal_high, 'y--', label='Normal (High)')
        axes[0, 1].plot(self.ph_range, self.ph_bahaya_high, 'r--', label='Bahaya (High)')
        axes[0, 1].set_title('pH Level')
        axes[0, 1].legend(fontsize=8)

        # Plot TDS
        axes[1, 0].plot(self.tds_range, self.tds_ideal, 'g-', label='Ideal')
        axes[1, 0].plot(self.tds_range, self.tds_normal, 'y-', label='Normal')
        axes[1, 0].plot(self.tds_range, self.tds_bahaya, 'r-', label='Bahaya')
        axes[1, 0].set_title('TDS (ppm)')
        axes[1, 0].legend()

        # Plot Temp
        axes[1, 1].plot(self.temp_range, self.temp_bahaya_low, 'r-', label='Bahaya (Low)')
        axes[1, 1].plot(self.temp_range, self.temp_normal_low, 'y-', label='Normal (Low)')
        axes[1, 1].plot(self.temp_range, self.temp_ideal, 'g-', label='Ideal')
        axes[1, 1].plot(self.temp_range, self.temp_normal_high, 'y--', label='Normal (High)')
        axes[1, 1].plot(self.temp_range, self.temp_bahaya_high, 'r--', label='Bahaya (High)')
        axes[1, 1].set_title('Temperature (°C)')
        axes[1, 1].legend(fontsize=8)

        for ax in axes.flat:
            ax.grid(True, alpha=0.3)
            ax.set_ylim([0, 1.05])

        plt.tight_layout()
        plt.show()

visualizer = FuzzyWaterQualityVisualization()
visualizer.plot_all_membership_functions()

**Strategi Penyeimbangan Data (Data Balancing)**

Setelah proses pelabelan menggunakan logika fuzzy, distribusi data seringkali menjadi tidak seimbang (*imbalanced*), di mana satu kelas jauh lebih dominan dibandingkan kelas lainnya. Masalah ini dapat ditangani dengan pendekatan hibrida:

*   **Train-Test Split:** Data dipisahkan menjadi 80% untuk pelatihan dan 20% untuk pengujian. Penting untuk dicatat bahwa proses penyeimbangan hanya dilakukan pada data pelatihan agar data pengujian tetap murni.
*   **SMOTE (Synthetic Minority Over-sampling Technique):** teknik menciptakan sampel baru secara sintetis di antara data minoritas dan tetangga terdekatnya. Hasilnya jauh lebih efektif daripada duplikasi data karena berhasil memperkaya variasi fitur.
*   **Random UnderSampler:** teknik mengurangi jumlah sampel pada kelas mayoritas secara acak hingga mencapai nilai median yang telah ditentukan. Hal ini mencegah model menjadi terlalu 'malas' karena terlalu sering melihat pola kelas yang sama.

Hasil akhirnya adalah dataset pelatihan yang netral, stabil, dan memberikan kesempatan yang adil bagi model untuk mempelajari karakteristik setiap status kualitas air.

In [ ]:
# ====== BALANCING DATASET ======
print("==== MENYEIMBANGKAN DATASET ====")

# Definisikan 4 fitur utama yang akan digunakan
feature_columns = ['DO(mg/l)', 'pH(ph_units)', 'TDS(ppm)', 'Temp(cel)']
X = df[feature_columns].values
y = df['status'].values
print(f"Fitur yang digunakan: {feature_columns}")
print(f"Jumlah fitur: {len(feature_columns)}")
print(f"Total sampel: {len(X)}")

urutan_label = ['ideal', 'normal', 'bahaya']

# ====== TRAIN TEST SPLIT (SEBELUM BALANCING) ======
print("\n======== PERSIAPAN DATA - TRAIN TEST SPLIT (SEBELUM BALANCING) ========")
X_train_raw, X_test, y_train_raw, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"✅ Data Training (sebelum balancing): {len(X_train_raw)} sampel")
print(f"✅ Data Testing (asli, TIDAK di-SMOTE): {len(X_test)} sampel")
print("   -> Test set ini akan tetap mencerminkan distribusi data asli/")
print("      data real-time, sehingga akurasi yang dilaporkan lebih jujur.\n")

# Tampilkan distribusi awal (pada training set saja)
print("📊 Distribusi Kelas Training SEBELUM Balancing:")
initial_distribution = {}
for label in urutan_label:
    initial_distribution[label] = np.sum(y_train_raw == label)

for label in urutan_label:
    count = initial_distribution.get(label, 0)
    print(f"  {label:12s}: {count:5d} sampel ({count/len(y_train_raw)*100:5.2f}%)")
print(f"  {'TOTAL':12s}: {len(y_train_raw):5d} sampel")

# Hitung median sebagai target
jumlah_per_kelas = [initial_distribution[label] for label in urutan_label]
target_median = int(np.median(jumlah_per_kelas))

print(f"\n🎯 Median jumlah sampel per kelas (training): {target_median}")

# Strategi balancing
over_strategy = {}
under_strategy = {}

for label in urutan_label:
    count = initial_distribution[label]
    if count < target_median:
        over_strategy[label] = target_median
    elif count > target_median:
        under_strategy[label] = target_median

# Build pipeline
steps = []

# Tambahkan oversampling jika ada kelas minority
if over_strategy:
    steps.append(('oversample', SMOTE(
        sampling_strategy=over_strategy,
        random_state=42,
        k_neighbors=min(3, min(initial_distribution.values()) - 1)
    )))

# Tambahkan undersampling jika ada kelas majority
if under_strategy:
    steps.append(('undersample', RandomUnderSampler(
        sampling_strategy=under_strategy,
        random_state=42
    )))

if steps:
    pipeline = Pipeline(steps)
    print("⏳ Proses balancing dinamis (hanya pada training set)...")
    X_train, y_train = pipeline.fit_resample(X_train_raw, y_train_raw)
else:
    print("⚠️ Tidak diperlukan balancing (semua kelas training sudah seimbang).")
    X_train, y_train = X_train_raw.copy(), y_train_raw.copy()

# Tampilkan distribusi setelah balancing
print("\n✅ Distribusi Kelas Training SETELAH Balancing:")
final_distribution = {}
for label in urutan_label:
    final_distribution[label] = np.sum(y_train == label)

for label in urutan_label:
    count = final_distribution.get(label, 0)
    print(f"  {label:12s}: {count:5d} sampel ({count/len(y_train)*100:5.2f}%)")
print(f"  {'TOTAL':12s}: {len(y_train):5d} sampel")

**Pengembangan Model, Tuning, dan Evaluasi KNN**

Tahap ini adalah inti dari kecerdasan buatan dalam sistem ini, di mana data yang telah diseimbangkan diproses untuk menghasilkan model prediksi:

*   **Standard Scaling:** Menyamakan skala semua fitur (0-1 atau Z-score) agar fitur dengan nilai besar (seperti TDS dalam ratusan) tidak mendominasi fitur dengan nilai kecil (seperti pH) dalam perhitungan jarak Euclidean KNN.
*   **Hyperparameter Tuning:** Melakukan optimasi nilai K (tetangga terdekat) menggunakan *5-Fold Cross Validation* dari rentang K=3 hingga K=15 untuk menemukan titik akurasi tertinggi.
*   **Analisis Learning Curve:** Bagian krusial untuk memvalidasi stabilitas model. Kurva ini memvisualisasikan perbandingan skor pelatihan (*Training Score*) dan skor validasi (*Cross-validation Score*) seiring bertambahnya jumlah data.
    *   Jika kedua kurva berdekatan dan menuju nilai akurasi tinggi, model dianggap memiliki generalisasi yang baik.
    *   Digunakan untuk mendeteksi apakah model mengalami *High Bias* (Underfitting) atau *High Variance* (Overfitting).
*   **Metrik Evaluasi:** Menggunakan *Confusion Matrix* untuk melihat detail prediksi benar/salah pada setiap kelas, serta *Weighted F1-Score* sebagai metrik utama karena memberikan bobot yang adil berdasarkan proporsi kelas asli.

In [ ]:
# ====== NORMALISASI (FIT HANYA PADA TRAINING) ======
print("\n======== NORMALISASI DATA ========")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Data berhasil dinormalisasi (4 fitur)")
print(f"   Feature columns: {feature_columns}")

# ====== VALIDASI SILANG K-FOLD ======
print("\n======== TUNING HYPERPARAMETER (K-Nearest Neighbors) ========")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
k_values = list(range(3, 16, 2))
cv_scores = []
cv_stds = []
print("Mencari nilai K optimal dengan 5-Fold Cross Validation...")

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, weights='uniform', metric='euclidean')
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    mean_score = scores.mean()
    std_score = scores.std()

    cv_scores.append(mean_score)
    cv_stds.append(std_score)

    print(f"K={k:2d} | Mean CV Accuracy = {mean_score:.4f} | Std = {std_score:.4f}")

best_k = k_values[np.argmax(cv_scores)]
best_acc = max(cv_scores)

print(f"🎯 Nilai K Optimal: {best_k}")
print(f"🎯 Best CV Accuracy: {best_acc:.4f} ({best_acc*100:.2f}%)")

# ====== EVALUASI MODEL ======
best_model = KNeighborsClassifier(n_neighbors=best_k, weights='uniform', metric='euclidean')
best_model.fit(X_train_scaled, y_train)
y_pred = best_model.predict(X_test_scaled)
test_accuracy = accuracy_score(y_test, y_pred)
weighted_f1 = f1_score(y_test, y_pred, average='weighted')
macro_f1 = f1_score(y_test, y_pred, average='macro')

print("\n======== EVALUASI MODEL ========")
print(f"🚀 TRAINING MODEL DENGAN K={best_k}")
print(f"🎯 Test Accuracy      = {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"🎯 Weighted F1-Score  = {weighted_f1:.4f}")
print(f"🎯 Macro F1-Score     = {macro_f1:.4f}")

# Classification Report
print("\n======== Classification Report: ========")
print(classification_report(y_test, y_pred, labels=urutan_label, target_names=urutan_label, zero_division=0))
cm = confusion_matrix(y_test, y_pred, labels=urutan_label)
print("\nActual \\ Predicted:", end="")
for label in urutan_label:
    print(f"{label:>12}", end="")
print()
print("-" * 80)
for i, label in enumerate(urutan_label):
    print(f"{label:>18}", end="")
    for j in range(len(urutan_label)):
        print(f"{cm[i][j]:>12}", end="")
    print()

# ====== ANALISIS LEARNING CURVE ======
print("\n======== 📈 ANALISIS LEARNING CURVE ========")

train_sizes, train_scores, test_scores = learning_curve(
    best_model, X_train_scaled, y_train, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy', random_state=42
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

plt.figure(figsize=(6, 4))
plt.plot(train_sizes, train_mean, 'o-', color='blue', linewidth=2,
         markersize=6, label='Training Score')
plt.plot(train_sizes, test_mean, 'o-', color='green', linewidth=2,
         markersize=6, label='Cross-validation Score')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                 alpha=0.15, color='blue')
plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std,
                 alpha=0.15, color='green')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xlabel('Training Examples', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Learning Curve - KNN Classifier (Data Balanced)', fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=11)
plt.tight_layout()
plt.show()

# Analisis hasil learning curve
print("\n🔍 Analisis Learning Curve:")
gap = train_mean[-1] - test_mean[-1]
print(f"  - Training Score (final): {train_mean[-1]:.4f}")
print(f"  - CV Score (final):       {test_mean[-1]:.4f}")
print(f"  - Gap:                    {gap:.4f}")

if test_mean[-1] > 0.9 and train_mean[-1] > 0.9 and gap < 0.05:
    performance = "✅ Model terlihat SANGAT BAIK (high performance, low overfitting)"
elif gap > 0.1:
    performance = "⚠️ Model mungkin OVERFITTING (training score >> CV score)"
elif train_mean[-1] < 0.8:
    performance = "⚠️ Model mungkin UNDERFITTING (perlu model lebih kompleks)"
elif test_mean[-1] > 0.85:
    performance = "✅ Model memiliki performa BAIK dan SEIMBANG"
else:
    performance = "⚠️ Model memiliki performa CUKUP (bisa ditingkatkan)"

print(f"  Kesimpulan: {performance}")

**Perancangan Sistem Inferensi dan Validasi Sensor (Deployment Ready)**

Bagian ini mengubah model matematika yang abstrak menjadi fungsi praktis yang siap diimplementasikan pada sistem monitoring real-time. Fokus utamanya adalah keandalan input:

*   **Validasi Rentang Fisik:** Sebelum melakukan prediksi, sistem memeriksa apakah angka yang diterima dari sensor masuk akal (misal: pH tidak mungkin di bawah 0 atau di atas 14). Jika sensor mengirim data noise/error, sistem akan mendeteksinya.
*   **Fungsi `prediksi_kualitas_air`:** Fungsi terintegrasi ini menggabungkan tahap normalisasi (scaler) dan prediksi KNN. Hasil akhirnya bukan hanya label (Ideal/Normal/Bahaya), tetapi juga nilai probabilitas yang menunjukkan tingkat keyakinan model terhadap kondisi air tersebut.

In [ ]:
# FUNGSI VALIDASI INPUT SENSOR UNTUK INFERENCE (DATA REAL-TIME)
RENTANG_VALID_SENSOR = {
    'DO(mg/l)':     (0.0, 20.0),
    'pH(ph_units)': (0.0, 14.0),
    'TDS(ppm)':     (0.0, 2000.0),
    'Temp(cel)':    (0.0, 50.0),
}

def validasi_dan_siapkan_input(do, ph, tds, temp):
    nilai_mentah = {
        'DO(mg/l)': do, 'pH(ph_units)': ph,
        'TDS(ppm)': tds, 'Temp(cel)': temp,
    }
    pesan = []
    ada_error = False
    nilai_valid = {}

    for kolom, nilai in nilai_mentah.items():
        lo, hi = RENTANG_VALID_SENSOR[kolom]
        if nilai is None or (isinstance(nilai, float) and np.isnan(nilai)):
            pesan.append(f"{kolom}: nilai kosong/NaN, kemungkinan sensor terputus")
            ada_error = True
            nilai_valid[kolom] = (lo + hi) / 2  # nilai netral sementara
            continue
        if nilai < lo or nilai > hi:
            pesan.append(
                f"{kolom}: nilai {nilai} di luar rentang fisik wajar "
                f"[{lo}, {hi}], kemungkinan sensor error/belum terkalibrasi"
            )
            ada_error = True
        nilai_valid[kolom] = float(np.clip(nilai, lo, hi))

    status_sensor = 'sensor_error' if ada_error else 'ok'
    return nilai_valid, status_sensor, pesan

def prediksi_kualitas_air(do, ph, tds, temp, model, scaler, feature_columns):
    nilai_valid, status_sensor, pesan = validasi_dan_siapkan_input(do, ph, tds, temp)

    if status_sensor == 'sensor_error':
        return {
            'status': 'sensor_error',
            'label_prediksi': None,
            'pesan': pesan,
            'catatan': 'Prediksi tidak dijalankan karena pembacaan sensor '
                       'tidak wajar. Periksa kondisi fisik sensor sebelum '
                       'mempercayai status kualitas air yang dihasilkan.'
        }

    X_input = np.array([[nilai_valid[col] for col in feature_columns]])
    X_input_scaled = scaler.transform(X_input)
    label_prediksi = model.predict(X_input_scaled)[0]
    proba = model.predict_proba(X_input_scaled)[0]
    return {
        'status': 'ok',
        'label_prediksi': label_prediksi,
        'probabilitas': dict(zip(model.classes_, proba.round(4).tolist())),
        'nilai_yang_dipakai': nilai_valid,
        'pesan': pesan,
    }

print("✅ Fungsi validasi_dan_siapkan_input() dan prediksi_kualitas_air()")
print("   siap dipakai untuk inference pada data sensor real-time.")

**Evaluasi Model pada Data Tak Terlihat dan Simulasi Skenario**

Langkah ini bertujuan untuk menguji sejauh mana model mampu mempertahankan akurasinya ketika dihadapkan pada data yang belum pernah dipelajari sebelumnya (data yang tidak masuk dalam proses training maupun SMOTE).

*   **Uji Sampel Acak:** Mengambil sampel dari data uji asli untuk memverifikasi kesesuaian antara label aktual (dari logika fuzzy) dengan prediksi model KNN.
*   **Simulasi Skenario Kritis:** Pengujian khusus menggunakan data buatan (sintetis) untuk mensimulasikan kondisi ekstrem seperti penurunan oksigen secara mendadak atau lonjakan pH. Ini penting untuk memastikan sistem peringatan dini (*early warning system*) berfungsi.
*   **Uji Robustness (Noise Injection):** Memberikan sedikit gangguan (*noise*) pada nilai sensor untuk melihat apakah model tetap stabil atau malah berubah drastis. Model yang baik harus tetap konsisten meskipun ada fluktuasi kecil pada pembacaan sensor.

In [ ]:
# ====== MELIHAT EVALUASI MODEL & PENGUJIAN DENGAN DATA YANG TAK PERNAH DILIHAT ======
print("\n🔹 Contoh prediksi pada 10 sampel acak dari data uji:")
indices = np.random.choice(len(X_test), size=10, replace=False)
for idx in indices:
    actual = y_test[idx]
    pred = y_pred[idx]
    features = X_test[idx]
    status = "✅" if actual == pred else "❌"
    print(f"   {status} Sampel ke-{idx:4d} | Aktual: {actual:7} | Prediksi: {pred:7} | "
          f"DO={features[0]:.2f}, pH={features[1]:.2f}, TDS={features[2]:.1f}, Temp={features[3]:.2f}")

# 7. Contoh inferensi untuk data sensor real-time (menggunakan fungsi yang sudah ada)
print("\n🔹 Contoh inferensi untuk data sensor (3 skenario):")

# Skenario 1: Data ideal
do, ph, tds, temp = 7.5, 6.8, 500, 26.0
hasil = prediksi_kualitas_air(do, ph, tds, temp, best_model, scaler, feature_columns)
print(f"\n   Skenario 1 - Ideal (DO={do}, pH={ph}, TDS={tds}, Temp={temp})")
print(f"   Status sensor: {hasil['status']}")
if hasil['status'] == 'ok':
    print(f"   Prediksi: {hasil['label_prediksi']}")
    print(f"   Probabilitas: {hasil['probabilitas']}")
else:
    print(f"   Pesan error: {hasil['pesan']}")

# Skenario 2: Data bahaya (DO rendah)
do, ph, tds, temp = 2.5, 7.0, 800, 28.0
hasil = prediksi_kualitas_air(do, ph, tds, temp, best_model, scaler, feature_columns)
print(f"\n   Skenario 2 - Bahaya (DO={do}, pH={ph}, TDS={tds}, Temp={temp})")
print(f"   Status sensor: {hasil['status']}")
if hasil['status'] == 'ok':
    print(f"   Prediksi: {hasil['label_prediksi']}")
    print(f"   Probabilitas: {hasil['probabilitas']}")
else:
    print(f"   Pesan error: {hasil['pesan']}")

# Skenario 3: Data sensor error (pH di luar rentang)
do, ph, tds, temp = 6.0, 12.0, 600, 27.0
hasil = prediksi_kualitas_air(do, ph, tds, temp, best_model, scaler, feature_columns)
print(f"\n   Skenario 3 - Sensor error (DO={do}, pH={ph}, TDS={tds}, Temp={temp})")
print(f"   Status sensor: {hasil['status']}")
if hasil['status'] == 'ok':
    print(f"   Prediksi: {hasil['label_prediksi']}")
    print(f"   Probabilitas: {hasil['probabilitas']}")
else:
    print(f"   Pesan error: {hasil['pesan']}")

print("\n📌 PENGUJIAN DENGAN DATA SINTETIS (kombinasi nilai baru)")
sampel_baru = [
    # [DO, pH, TDS, Temp, deskripsi]
    [8.0, 7.0, 600, 27.0, "Ideal sempurna"],
    [5.5, 6.5, 900, 29.0, "Normal mendekati ideal"],
    [3.5, 6.0, 1100, 25.0, "DO rendah, normal lainnya"],
    [2.0, 7.5, 800, 28.0, "DO sangat rendah (bahaya)"],
    [7.0, 5.0, 700, 26.0, "pH asam (bahaya)"],
    [7.0, 9.0, 700, 26.0, "pH basa (bahaya)"],
    [7.0, 7.0, 1500, 26.0, "TDS tinggi (bahaya)"],
    [7.0, 7.0, 700, 36.0, "Suhu tinggi (bahaya)"],
    [7.0, 7.0, 700, 18.0, "Suhu rendah (bahaya)"],
    [4.0, 6.0, 1000, 32.0, "DO rendah suhu tinggi (bahaya)"],
    [6.0, 7.0, 1000, 26.0, "Normal"],
]

print("-" * 100)
print(f"{'No':<4} {'Deskripsi':<30} {'DO':<6} {'pH':<6} {'TDS':<8} {'Temp':<6} {'Prediksi':<12} {'Probabilitas (Ideal,Normal,Bahaya)'}")
print("-" * 100)

for i, sampel in enumerate(sampel_baru, 1):
    do, ph, tds, temp, deskripsi = sampel
    hasil = prediksi_kualitas_air(do, ph, tds, temp, best_model, scaler, feature_columns)
    if hasil['status'] == 'ok':
        pred = hasil['label_prediksi']
        prob = hasil['probabilitas']
        prob_str = f"({prob.get('ideal',0):.3f}, {prob.get('normal',0):.3f}, {prob.get('bahaya',0):.3f})"
        print(f"{i:<4} {deskripsi:<30} {do:<6.1f} {ph:<6.1f} {tds:<8.1f} {temp:<6.1f} {pred:<12} {prob_str}")
    else:
        print(f"{i:<4} {deskripsi:<30} {do:<6.1f} {ph:<6.1f} {tds:<8.1f} {temp:<6.1f} {'ERROR':<12} {str(hasil['pesan'])[:50]}...")

print("-" * 100)

# 3. Uji robustness dengan data noise (sedikit perubahan)
print("\n📌 UJI ROBUSTNESS: Pengaruh noise kecil pada prediksi")
# Ambil satu sampel ideal, lalu tambahkan noise kecil
sampel_ideal = [7.5, 6.8, 500, 26.0]
do, ph, tds, temp = sampel_ideal
print(f"   Sampel basis (Ideal): DO={do}, pH={ph}, TDS={tds}, Temp={temp}")
hasil_asli = prediksi_kualitas_air(do, ph, tds, temp, best_model, scaler, feature_columns)
print(f"   Prediksi basis: {hasil_asli['label_prediksi']} (prob: {hasil_asli['probabilitas']})")

# Tambahkan noise kecil (misal +- 5% pada setiap parameter)
noise_levels = [0.02, 0.05, 0.10]  # 2%, 5%, 10% noise
for noise in noise_levels:
    do_noise = do * (1 + np.random.uniform(-noise, noise))
    ph_noise = ph * (1 + np.random.uniform(-noise, noise))
    tds_noise = tds * (1 + np.random.uniform(-noise, noise))
    temp_noise = temp * (1 + np.random.uniform(-noise, noise))
    # Clip agar dalam rentang valid
    do_noise = np.clip(do_noise, 0, 20)
    ph_noise = np.clip(ph_noise, 0, 14)
    tds_noise = np.clip(tds_noise, 0, 2000)
    temp_noise = np.clip(temp_noise, 0, 50)

    hasil_noise = prediksi_kualitas_air(do_noise, ph_noise, tds_noise, temp_noise, best_model, scaler, feature_columns)
    if hasil_noise['status'] == 'ok':
        print(f"   Noise {noise*100:3.0f}% : DO={do_noise:.2f}, pH={ph_noise:.2f}, TDS={tds_noise:.1f}, Temp={temp_noise:.2f} -> Pred={hasil_noise['label_prediksi']} (prob: {hasil_noise['probabilitas']})")
    else:
        print(f"   Noise {noise*100:3.0f}% : ERROR - {hasil_noise['pesan']}")

**Manajemen Penyimpanan Hasil dan Dokumentasi Teknis (Google Drive)**

Bagian terakhir ini adalah tahap finalisasi di mana seluruh hasil eksperimen machine learning diekspor ke Google Drive agar sistem dapat diimplementasikan di perangkat lain tanpa perlu melatih ulang data. Proses yang dilakukan meliputi:

*   **Serialisasi Model Biner (.pkl):** Menggunakan library `joblib` untuk menyimpan objek model KNN, Scaler normalisasi, dan metadata fitur ke dalam format biner. Ini memungkinkan 'pembekuan' kecerdasan model agar siap dipanggil kembali kapan saja.
*   **Manajemen Dataset:** Menyimpan dua varian dataset dalam format CSV:
    *   **Dataset Balanced:** Data hasil SMOTE yang digunakan untuk audit pelatihan.
    *   **Dataset Test Original:** Data asli yang belum disentuh manipulasi, berfungsi sebagai tolak ukur performa murni.
*   **Otomatisasi Laporan Metadata:** Menghasilkan file teks (`model_metadata.txt`) yang berisi ringkasan teknis mulai dari parameter K optimal hingga Confusion Matrix secara detail.
*   **Ekspor Visualisasi Evaluasi:** Menyimpan grafik *Cross-Validation* dan *Confusion Matrix* dalam resolusi tinggi (300 DPI) untuk keperluan lampiran laporan atau presentasi ilmiah.

In [ ]:
# 🗂️ Tentukan folder penyimpanan
output_dir = '/content/drive/MyDrive/hasil_machine_learning/MODEL-AQUAPONICS/'
os.makedirs(output_dir, exist_ok=True)

# ====== SIMPAN SEMUA OUTPUT ======
print("\n" + "="*60)
print("💾 PENYIMPANAN HASIL KE GOOGLE DRIVE")
print("="*60)

# 1. Simpan Model KNN
model_path = os.path.join(output_dir, 'knn_water_quality_model.pkl')
joblib.dump(best_model, model_path)
print(f"✅ Model KNN disimpan: {model_path}")

# 2. Simpan Scaler (4 FITUR)
scaler_path = os.path.join(output_dir, 'scaler.pkl')
joblib.dump(scaler, scaler_path)

# 3. Simpan Feature Columns (4 FITUR)
features_path = os.path.join(output_dir, 'feature_columns.pkl')
joblib.dump(feature_columns, features_path)
print(f"✅ Feature columns disimpan: {features_path}")

# 4. Simpan Dataset dengan Status (4 fitur)
# 4a. Dataset training (sudah balanced/di-SMOTE)
dataset_path = os.path.join(output_dir, 'dataset_balanced_4features.csv')
df_balanced = pd.DataFrame(X_train, columns=feature_columns)
for col in feature_columns:
    if col in df_balanced.columns:
        df_balanced[col] = df_balanced[col].astype(float).round(2)
df_balanced['status'] = y_train
df_balanced.to_csv(dataset_path, index=False)
print(f"✅ Dataset training balanced disimpan: {dataset_path}")

# 4b. Dataset test (data ASLI, tidak pernah disentuh SMOTE) -- disimpan
dataset_test_path = os.path.join(output_dir, 'dataset_test_original.csv')
df_test_original = pd.DataFrame(X_test, columns=feature_columns)
for col in feature_columns:
    if col in df_test_original.columns:
        df_test_original[col] = df_test_original[col].astype(float).round(2)
df_test_original['status'] = y_test
df_test_original.to_csv(dataset_test_path, index=False)
print(f"✅ Dataset test asli (non-SMOTE) disimpan: {dataset_test_path}")

# 5. Simpan Metadata
metadata_path = os.path.join(output_dir, 'model_metadata.txt')
with open(metadata_path, 'w', encoding='utf-8') as f:
    f.write("📊 INFORMASI FITUR\n")
    f.write("-" * 60 + "\n")
    f.write(f"Jumlah Fitur: {len(feature_columns)}\n")
    f.write(f"Nama Fitur: {', '.join(feature_columns)}\n\n")

    f.write("📊 INFORMASI DATASET\n")
    f.write("-" * 60 + "\n")
    f.write(f"Total Data (Original)                   : {len(df)} sampel\n")
    f.write(f"Total Data Training (Balanced)          : {len(y_train)} sampel\n")
    f.write(f"Training Set                            : {len(y_train)} sampel\n")
    f.write(f"Testing Set (data asli, tidak di-SMOTE) : {len(y_test)} sampel\n\n")
    f.write("\n")

    f.write("🔍 PARAMETER MODEL\n")
    f.write("-" * 60 + "\n")
    f.write(f"Algoritma       : K-Nearest Neighbors (KNN)\n")
    f.write(f"Nilai K Optimal : {best_k}\n")
    f.write(f"Weights         : uniform\n")
    f.write(f"Metric          : euclidean\n")

    f.write("\n📈 PERFORMA MODEL\n")
    f.write("-" * 60 + "\n")
    f.write(f"Best CV Accuracy    : {best_acc:.4f} ({best_acc*100:.2f}%)\n")
    f.write(f"Test Accuracy       : {test_accuracy:.4f} ({test_accuracy*100:.2f}%)\n")
    f.write(f"Weighted F1-Score   : {weighted_f1:.4f}\n")
    f.write(f"Macro F1-Score      : {macro_f1:.4f}\n\n")

    f.write("📊 CONFUSION MATRIX\n")
    f.write("-" * 60 + "\n")
    cm = confusion_matrix(y_test, y_pred, labels=urutan_label)
    f.write("Actual\\Predicted" + "".join([f"{l:>10}" for l in urutan_label]) + "\n")
    for i, label in enumerate(urutan_label):
        f.write(f"{label:12s}" + "".join([f"{cm[i,j]:>10}" for j in range(len(urutan_label))]) + "\n")

print(f"✅ Metadata disimpan: {metadata_path}")

# Simpan Visualisasi
# Visualisasi KNN Cross-Validation Results
plt.figure(figsize=(10, 6))
plt.errorbar(k_values, cv_scores, yerr=cv_stds, fmt='o-', capsize=5,
             color='blue', ecolor='red', linewidth=2, markersize=8)
plt.axhline(y=max(cv_scores), color='green', linestyle='--', alpha=0.7, linewidth=2)
plt.grid(True, linestyle='--', alpha=0.7)
plt.xlabel('Nilai K', fontsize=12)
plt.ylabel('Accuracy (5-Fold CV)', fontsize=12)
plt.title('Performa KNN terhadap Nilai K (Data Balanced)', fontsize=14, fontweight='bold')
plt.xticks(k_values)
plt.scatter(best_k, best_acc, color='red', s=200, zorder=5,
            label=f'Best K={best_k} (Acc={best_acc:.4f})', marker='*')
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'knn_cv_results_balanced.png'), dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Grafik CV results disimpan")

# Visualisasi Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=urutan_label)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=urutan_label, yticklabels=urutan_label,
            annot_kws={"size": 14, "weight": "bold"},
            cbar_kws={'label': 'Jumlah Sampel'},
            linewidths=1, linecolor='gray')
plt.xlabel('Predicted', fontsize=14, fontweight='bold')
plt.ylabel('Actual', fontsize=14, fontweight='bold')
plt.title('Confusion Matrix (Test Set - Data Balanced)', fontsize=16, fontweight='bold')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'confusion_matrix_balanced.png'), dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Grafik confusion matrix disimpan")

# Ringkasan File yang Tersimpan (Fix typo 1024*1024)
print("\n" + "="*60)
print("🗂️ DAFTAR FILE TERSIMPAN")
print("="*60)
saved_files = [
    ('Model KNN', 'knn_water_quality_model.pkl'),
    ('Scaler', 'scaler.pkl'),
    ('Feature Columns', 'feature_columns.pkl'),
    ('Dataset Training Balanced', 'dataset_balanced_4features.csv'),
    ('Dataset Test asli ', 'dataset_test_original.csv'),
    ('Metadata', 'model_metadata.txt'),
    ('Grafik CV Results', 'knn_cv_results_balanced.png'),
    ('Grafik Caonfusion Matrix', 'confusion_matrix_balanced.png'),
]
for name, filename in saved_files:
    filepath = os.path.join(output_dir, filename)
    if os.path.exists(filepath):
        size = os.path.getsize(filepath)
        size_str = f"{size/1024:.2f} KB" if size < 1024*1024 else f"{size/(1024*1024):.2f} MB" # Fix typo
        print(f"✅ {name:25s}: {filename:40s} ({size_str})")
    else:
        print(f"❌ {name:25s}: {filename:40s} (TIDAK DITEMUKAN)")

**Dengan selesainya tahap ini, sistem monitoring kualitas air berbasis KNN telah resmi terdokumentasi dan siap untuk tahap integrasi perangkat keras.**

#**AUTO FILL DATASET**